# L12 — Project 6: Gesture-Cued Lighting

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yanluo/stem-on-stage-notebooks/blob/main/L12/06_gesture_cued_lighting/gesture_lighting_starter.ipynb)

**Goal:** train a tiny gesture classifier in Colab, export it as plain Python `if/elif`, paste it into a wearable micro:bit program, and have the dancer's gestures cue lantern color changes via radio.

**This is the bridge between L11 (machine learning on a laptop) and the lantern stack from L4–L7 (radio + state machines).** Every other L12 project does *analysis*. This one does *deployment* — the model leaves Colab and runs on a battery-powered board strapped to a dancer.

**Heads-up.** The notebook here is the *training* side. The device side is [`wearable_gesture.py`](wearable_gesture.py) — you'll paste the `predict(...)` function this notebook prints into that file.

> **New to pandas / numpy / sklearn?** Skim [`L12/00_python_data_tools/python_data_tools_starter.ipynb`](../00_python_data_tools/python_data_tools_starter.ipynb) first — it's a 30-minute tour of every function this notebook uses, with tiny standalone examples.

## Step 0 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

## Step 1 — Train the L11 model again (recap)

We reuse the still/walk/jump dataset from L11 to keep this notebook self-contained. To train on *your own* gestures, swap the URL for your captured CSV — every step below stays identical.

In [ ]:
SAMPLE_URL = "https://raw.githubusercontent.com/yanluo/stem-on-stage-notebooks/main/data/sample-still-walk-jump.csv"

if IN_COLAB:
    df = pd.read_csv(SAMPLE_URL)
else:
    df = pd.read_csv("../../data/sample-still-walk-jump.csv")

df["mag"] = np.sqrt(df["x"]**2 + df["y"]**2 + df["z"]**2)

def label_for(t):
    if t < 5:   return "still"
    if t < 15:  return "walk"
    if t < 20:  return "still"
    if t < 30:  return "jump"
    return "still"

df["label"] = df["t"].apply(label_for)

WINDOW_S, HZ = 1.0, 20
W = int(WINDOW_S * HZ)
rows = []
for start in range(0, len(df) - W, W):
    chunk = df.iloc[start:start + W]
    rows.append({
        "mean_mag": chunk["mag"].mean(),
        "std_mag":  chunk["mag"].std(),
        "max_mag":  chunk["mag"].max(),
        "range_x":  chunk["x"].max() - chunk["x"].min(),
        "range_y":  chunk["y"].max() - chunk["y"].min(),
        "range_z":  chunk["z"].max() - chunk["z"].min(),
        "label":    chunk["label"].mode().iloc[0],
    })
features_df = pd.DataFrame(rows)
FEATURES = ["mean_mag", "std_mag", "max_mag", "range_x", "range_y", "range_z"]

X_train, X_test, y_train, y_test = train_test_split(
    features_df[FEATURES], features_df["label"],
    test_size=0.3, random_state=42, stratify=features_df["label"])

clf = DecisionTreeClassifier(max_depth=3, random_state=42)
clf.fit(X_train, y_train)
print(f"train {clf.score(X_train, y_train):.3f}   test {clf.score(X_test, y_test):.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
plot_tree(clf, feature_names=FEATURES, class_names=clf.classes_,
          filled=True, rounded=True, fontsize=9, ax=ax)
plt.show()

## Step 2 — The trick: walk the tree, print Python

A decision tree is just nested `if/else`. Sklearn stores the tree as flat arrays under `clf.tree_`. The function below recurses through them and prints a Python `predict(...)` function that does exactly what `clf.predict(...)` does — in plain Python, with no imports.

**The output is what you paste into [`wearable_gesture.py`](wearable_gesture.py).**

In [ ]:
def tree_to_python(clf, feature_names, function_name="predict"):
    tree = clf.tree_
    classes = clf.classes_
    lines = [f"def {function_name}({', '.join(feature_names)}):"]

    def recurse(node, depth):
        indent = "    " * (depth + 1)
        if tree.feature[node] == _tree.TREE_UNDEFINED:
            class_idx = int(np.argmax(tree.value[node][0]))
            lines.append(f'{indent}return "{classes[class_idx]}"')
            return
        name = feature_names[tree.feature[node]]
        thresh = tree.threshold[node]
        lines.append(f"{indent}if {name} <= {thresh:.2f}:")
        recurse(tree.children_left[node],  depth + 1)
        lines.append(f"{indent}else:")
        recurse(tree.children_right[node], depth + 1)

    recurse(0, 0)
    return "\n".join(lines)

src = tree_to_python(clf, FEATURES)
print(src)

## Step 3 — Sanity-check the exported predict() against sklearn

If the export is correct, running the printed function on every test row should match `clf.predict(...)` exactly.

In [ ]:
# exec() the source we just generated so we can call it as predict(...)
ns = {}
exec(src, ns)
predict = ns["predict"]

manual = [predict(*row) for row in X_test.itertuples(index=False)]
sklearn_preds = list(clf.predict(X_test))
print("all match:", manual == sklearn_preds)
print("first 5  manual:", manual[:5])
print("first 5 sklearn:", sklearn_preds[:5])

## Step 4 — Deploy

1. **Copy** the `def predict(...)` block printed in Step 2.
2. Open [`wearable_gesture.py`](wearable_gesture.py) and paste it into the marked spot (between the `# ----- PASTE FROM NOTEBOOK -----` markers).
3. Open [https://makecode.microbit.org/](https://makecode.microbit.org/), switch to Python view, paste the whole `wearable_gesture.py` file, and **Download** to flash a battery-powered micro:bit.
4. Strap the board to a dancer's wrist. The board now reads accelerometer at 20 Hz, computes features over a 1-second sliding window, runs your `predict(...)` on each window, and broadcasts `radio.send_string("color:still")` / `"color:walk"` / `"color:jump"` whenever the predicted label changes.
5. Wire the radio messages into the L7 lantern controller — when it sees `color:walk`, it changes pattern.

## What to build next (L13 starting goals)

1. **Train on *your* gestures.** Capture a recording with `arms-up`, `clap`, `twirl` (whatever maps to lantern moods you want). Re-run the notebook on it. Re-paste the new `predict()`. The wearable file doesn't change.
2. **Hysteresis.** Only fire a radio cue if the same label is predicted for 3 windows in a row — otherwise it'll flicker between classes during transitions.
3. **Pick a richer color rule.** Each label can drive both a NeoPixel color *and* a pattern (e.g., `twirl` → chase pattern, `clap` → flash).
4. **Compare against L10.** L10 used a hand-tuned threshold (`if mag > 2200: jump`). Run that detector and the trained tree side by side on the same recording. Which is more reliable?
5. **Confidence.** Replace `DecisionTreeClassifier` with `RandomForestClassifier` and gate radio cues on agreement among trees (e.g., 7 of 10 trees say `walk` → fire). Quieter but more reliable.

## Reflect (homework)

Write 3–4 sentences:
- What's the smallest set of gestures you'd start with for the showcase?
- The exported tree is `if/elif`. What does that buy you that, say, `joblib.dump(clf)` and loading it on the device wouldn't? (Hint: think about MicroPython.)
- What's one thing you predict will be hard about the device side?